# CrewAI Hello World

A modern intro to CrewAI: agents, tasks, crews, custom tools, and structured output.

## 1. Setup

In [2]:
import litellm
from importlib.metadata import version
print("litellm:", version("litellm"))          # 1.74.15.post2
from crewai.llm import LITELLM_AVAILABLE
print("LITELLM_AVAILABLE:", LITELLM_AVAILABLE)  # True




litellm: 1.74.15.post2
LITELLM_AVAILABLE: True


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # Needs OPENROUTER_API_KEY and SERPAPI_API_KEY in .env

True

In [4]:
from crewai import LLM

llm = LLM(
    model='openrouter/moonshotai/kimi-k2-thinking',
    api_key=os.getenv('OPENROUTER_API_KEY')
)

llm.model

'openrouter/moonshotai/kimi-k2-thinking'

## 2. Tools

We create a custom search tool using SerpAPI (reusing your existing `SERPAPI_API_KEY`).

In [5]:
from crewai.tools import tool
from langchain_community.utilities import SerpAPIWrapper

serpapi = SerpAPIWrapper(serpapi_api_key=os.getenv('SERPAPI_API_KEY'))

@tool("Web Search")
def search(query: str) -> str:
    """Search the web for real-time information about current events, people, weather, sports, etc."""
    return serpapi.run(query)

In [6]:
search.run(query='current Baltimore Ravens quarterback')

'[\'The Baltimore Ravens are a professional American football team based in Baltimore. The Ravens compete in the National Football League as a member of the American Football Conference North division. The team plays its home games at M&T Bank Stadium and is headquartered in Owings Mills, Maryland.\', \'Baltimore Ravens type: Football team.\', \'Baltimore Ravens entity_type: american_football, sports.\', \'Baltimore Ravens kgmid: /m/01ct6.\', \'Baltimore Ravens nfl_championships: 2013, 2001.\', \'Baltimore Ravens head_coach: Jesse Minter.\', \'Baltimore Ravens location: Baltimore, MD.\', \'Baltimore Ravens division: AFC North.\', \'Baltimore Ravens arena_stadium: M&T Bank Stadium.\', \'Baltimore Ravens backup_qb: Tyler Huntley baltimoreravens.com.\', \'Baltimore Ravens logo: Wordmark.\', \'Active ; Lamar Jackson. 8, QB ; John Jenkins. 94, DL ; Cornelius Johnson. 86, WR ; Carl Jones Jr. 48, LB ...\', \'Baltimore Ravens Depth Chart ; Lamar Jackson · Tyler Huntley · Joe Fagnano · Diego Pa

## 3. Agents

Agents are specialized AI entities with a `role`, `goal`, and `backstory` that shape their behavior.

In [12]:
from crewai import Agent

researcher = Agent(
    role='Senior Research Analyst',
    goal='Find comprehensive and accurate information about a given topic with a focus on recent developments',
    backstory='You are an experienced research analyst with a talent for finding relevant information and organizing it clearly.',
    tools=[search],
    llm=llm,
    verbose=True
)

writer = Agent(
    role='Content Writer',
    goal='Craft engaging, well-structured content based on research findings',
    backstory='You are a skilled writer with a passion for making complex topics accessible and engaging.',
    llm=llm,
    verbose=True
)

## 4. Tasks

Tasks define the specific work each agent performs. The `context` parameter lets tasks build on each other.

In [13]:
from crewai import Task

research_task = Task(
    description='Research the latest trends in the AI agent industry and provide a summary of the top 3 developments.',
    expected_output='A summary of the top 3 trending developments in the AI agent industry with a unique perspective on their significance.',
    agent=researcher
)

write_task = Task(
    description='Write an engaging 4-paragraph blog post about the AI agent industry based on the research findings.',
    expected_output='A 4-paragraph blog post in markdown format that is engaging, informative, and avoids complex jargon.',
    agent=writer,
    context=[research_task],
    output_file='blog-posts/new_post.md'
)

## 5. Crew + Kickoff

The Crew orchestrates agents and tasks. A sequential process runs tasks in order.

In [14]:
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True
)

In [15]:
result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2b698061-ad28-460f-bcc7-9006a01e9d97                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the latest trends in the AI agent industry and provide a summary of the top 3 developments.     │
│  ID: 11de96e9-49f4-48e1-8e00-58f354a6584d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Research Analyst                                                                                 │
│                                                                                                                 │
│  Task: Research the latest trends in the AI agent industry and provide a summary of the top 3 developments.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'AI agent industry latest trends 2024 developments'}                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool web_search executed with result: ['In 2024, AI agents are no longer a niche interest. Companies across industries are getting more serious about incorporating agents into their workflows - from ...', "The trend: Looking back on 2024,...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: Write an engaging 4-paragraph blog post about the AI agent industry based on the research findings.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # The Silent Revolution: How AI Agents Are Rewiring the Modern Workplace                                       │
│                                                                                                                 │
│  Something extraordinary is happening in offices around the world, and you might not even notice it. The AI     │
│  agent industry—once a playground for experimental prototypes—is quietly maturing into the backbone of how      │
│  modern businesses operate. This isn't just another tech trend; it's a fundamental rewiring of work itself,     │
│  driven by three powerful forces that are transforming agents from clever toys into essential infrastructure.   │
│                                                                                                                 │
│  First, the technology has become radically easier to deploy. When OpenAI, Microsoft, and Google released       │
│  their enterprise platforms this year, they didn't just launch new products—they created a new foundation.      │
│  Instead of building agents from scratch over months, companies can now assemble them like Lego blocks in       │
│  days. These platforms handle the complex stuff—memory, decision-making frameworks, and coordination—so that    │
│  non-technical teams can create specialized agents for finance, HR, or operations without writing a single      │
│  line of code. It's reminiscent of the cloud computing boom, but instead of renting server space, businesses    │
│  are now renting autonomous digital workers that snap together seamlessly.                                      │
│                                                                                                                 │
│  Second, we're witnessing the rise of AI teams that actually *team up*. The explosive 327% growth in            │
│  multi-agent workflows reflects a new reality: agents are no longer solo performers. They're forming            │
│  self-managing groups that can scope, plan, and execute entire projects with minimal human supervision. In      │
│  progressive organizations, AI agents now create the vast majority of databases and development environments,   │
│  but the real magic is in how they collaborate. A single product manager can orchestrate a dozen specialized    │
│  agents to handle competitor research, user interviews, prototyping, and testing simultaneously—compressing     │
│  months of work into days. The relationship between humans and AI has flipped: instead of people using AI       │
│  tools, we now have AI teams that occasionally check in with human advisors for strategic guidance.             │
│                                                                                                                 │
│  Most importantly, this shift is showing up in the bottom line. While the market's projected tenfold growth to  │
│  $50 billion by 2030 is impressive, the real story is in the accounting. AI agents have graduated from          │
│  experimental budgets to core operational spending because they deliver undeniable returns. Customer success    │
│  teams that once required fifteen people now run with five agents and two managers. Code deployment times have  │
│  dropped by 70%. Legal document reviews that consumed weeks now finish in hours. Companies aren't just adding   │
│  agents to old processes—they're rebuilding workflows a

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write an engaging 4-paragraph blog post about the AI agent industry based on the research findings.      │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2b698061-ad28-460f-bcc7-9006a01e9d97                                                                       │
│  Final Output: # The Silent Revolution: How AI Agents Are Rewiring the Modern Workplace                         │
│                                                                                                                 │
│  Something extraordinary is happening in offices around the world, and you might not even notice it. The AI     │
│  agent industry—once a playground for experimental prototypes—is quietly maturing into the backbone of how      │
│  modern businesses operate. This isn't just another tech trend; it's a fundamental rewiring of work itself,     │
│  driven by three powerful forces that are transforming agents from clever toys into essential infrastructure.   │
│                                                                                                                 │
│  First, the technology has become radically easier to deploy. When OpenAI, Microsoft, and Google released       │
│  their enterprise platforms this year, they didn't just launch new products—they created a new foundation.      │
│  Instead of building agents from scratch over months, companies can now assemble them like Lego blocks in       │
│  days. These platforms handle the complex stuff—memory, decision-making frameworks, and coordination—so that    │
│  non-technical teams can create specialized agents for finance, HR, or operations without writing a single      │
│  line of code. It's reminiscent of the cloud computing boom, but instead of renting server space, businesses    │
│  are now renting autonomous digital workers that snap together seamlessly.                                      │
│                                                                                                                 │
│  Second, we're witnessing the rise of AI teams that actually *team up*. The explosive 327% growth in            │
│  multi-agent workflows reflects a new reality: agents are no longer solo performers. They're forming            │
│  self-managing groups that can scope, plan, and execute entire projects with minimal human supervision. In      │
│  progressive organizations, AI agents now create the vast majority of databases and development environments,   │
│  but the real magic is in how they collaborate. A single product manager can orchestrate a dozen specialized    │
│  agents to handle competitor research, user interviews, prototyping, and testing simultaneously—compressing     │
│  months of work into days. The relationship between humans and AI has flipped: instead of people using AI       │
│  tools, we now have AI teams that occasionally check in with human advisors for strategic guidance.             │
│                                                                                                                 │
│  Most importantly, this shift is showing up in the bottom line. While the market's projected tenfold growth to  │
│  $50 billion by 2030 is impressive, the real story is in the accounting. AI agents have graduated from          │
│  experimental budgets to core operational spending because they deliver undeniable returns. Customer success    │
│  teams that once required fifteen people now run with five agents and two managers. Code deployment times have  │
│  dropped by 70%. Legal document reviews that consumed weeks now finish in hours. Companies aren't just adding   │
│  agents to old processes—they're rebuilding workflows 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
print(result.raw)

# The Silent Revolution: How AI Agents Are Rewiring the Modern Workplace

Something extraordinary is happening in offices around the world, and you might not even notice it. The AI agent industry—once a playground for experimental prototypes—is quietly maturing into the backbone of how modern businesses operate. This isn't just another tech trend; it's a fundamental rewiring of work itself, driven by three powerful forces that are transforming agents from clever toys into essential infrastructure.

First, the technology has become radically easier to deploy. When OpenAI, Microsoft, and Google released their enterprise platforms this year, they didn't just launch new products—they created a new foundation. Instead of building agents from scratch over months, companies can now assemble them like Lego blocks in days. These platforms handle the complex stuff—memory, decision-making frameworks, and coordination—so that non-technical teams can create specialized agents for finance, HR, or op

In [17]:
for i, task_output in enumerate(result.tasks_output):
    print(f'--- Task {i+1} ---')
    print(task_output.raw[:500])
    print()

--- Task 1 ---
The AI agent industry is experiencing a paradigm shift in 2024-2025, moving from experimental prototypes to enterprise-critical infrastructure. Here are the three most significant developments reshaping the landscape:

**1. Enterprise-Grade Agent Development Platforms Emerge as the New Operating System**

The release of OpenAI's Agents SDK and API, combined with Microsoft's evolution of Copilot into a full agent ecosystem and Google's Vertex AI Agentic Tools, represents more than just product la

--- Task 2 ---
# The Silent Revolution: How AI Agents Are Rewiring the Modern Workplace

Something extraordinary is happening in offices around the world, and you might not even notice it. The AI agent industry—once a playground for experimental prototypes—is quietly maturing into the backbone of how modern businesses operate. This isn't just another tech trend; it's a fundamental rewiring of work itself, driven by three powerful forces that are transforming agents from clever t

In [18]:
result.token_usage

UsageMetrics(total_tokens=9358, prompt_tokens=4872, cached_prompt_tokens=1024, completion_tokens=4486, reasoning_tokens=2282, cache_creation_tokens=0, successful_requests=8)

## 6. Structured Output

Use a Pydantic model to get typed, structured results from a task.

In [19]:
from pydantic import BaseModel
from typing import List


class TrendReport(BaseModel):
    topic: str
    trends: List[str]
    summary: str

In [20]:
structured_researcher = Agent(
    role='Trend Analyst',
    goal='Identify and summarize the top trends for a given topic',
    backstory='You are a data-driven analyst who excels at identifying patterns and summarizing them concisely.',
    tools=[search],
    llm=llm,
    verbose=True
)

structured_task = Task(
    description='Research the top 3 trends in large language models right now.',
    expected_output='A structured report with the topic, a list of 3 trends, and a brief overall summary.',
    agent=structured_researcher,
    output_pydantic=TrendReport
)

structured_crew = Crew(
    agents=[structured_researcher],
    tasks=[structured_task],
    process=Process.sequential,
    verbose=True
)

In [21]:
structured_result = structured_crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 32d1219e-5bf2-4377-bf10-42950feecc5c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the top 3 trends in large language models right now.                                            │
│  ID: f4195aeb-19b4-4116-b807-6a313780d9f3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│  Task: Research the top 3 trends in large language models right now.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  topic='Large Language Models' trends=['Multimodal Integration: LLMs are rapidly expanding beyond text to       │
│  seamlessly process images, audio, and video, with models like GPT-4V, Gemini, and open-source alternatives     │
│  demonstrating sophisticated vision-language understanding capabilities that enable new applications in         │
│  document analysis, visual reasoning, and interactive AI assistants.', 'Open-Source Competition: The            │
│  open-source community is closing the gap with proprietary models, with releases like Llama 3, Mixtral, and     │
│  DBRX offering near state-of-the-art performance while providing transparency, customizability, and cost        │
│  advantages, challenging the dominance of closed systems from OpenAI, Google, and Anthropic.', 'Efficiency and  │
│  Edge Optimization: Growing emphasis on model compression, quantization, and specialized hardware is making     │
│  LLMs more accessible, with trends toward smaller yet powerful models that can run on consumer hardware,        │
│  reduced inference costs, and improved inference speeds making deployment more practical for businesses and     │
│  individual developers.'] summary="The large language model landscape is rapidly evolving with three dominant   │
│  trends: multimodal capabilities that integrate vision and language, explosive growth in open-source models     │
│  challenging proprietary systems, and increasing focus on efficiency through model compression and edge         │
│  deployment. These trends reflect the industry's push toward more capable, accessible, and practical AI         │
│  systems."                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Research the top 3 trends in large language models right now.                                            │
│  Agent: Trend Analyst                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 32d1219e-5bf2-4377-bf10-42950feecc5c                                                                       │
│  Final Output: {"topic":"Large Language Models","trends":["Multimodal Integration: LLMs are rapidly expanding   │
│  beyond text to seamlessly process images, audio, and video, with models like GPT-4V, Gemini, and open-source   │
│  alternatives demonstrating sophisticated vision-language understanding capabilities that enable new            │
│  applications in document analysis, visual reasoning, and interactive AI assistants.","Open-Source              │
│  Competition: The open-source community is closing the gap with proprietary models, with releases like Llama    │
│  3, Mixtral, and DBRX offering near state-of-the-art performance while providing transparency,                  │
│  customizability, and cost advantages, challenging the dominance of closed systems from OpenAI, Google, and     │
│  Anthropic.","Efficiency and Edge Optimization: Growing emphasis on model compression, quantization, and        │
│  specialized hardware is making LLMs more accessible, with trends toward smaller yet powerful models that can   │
│  run on consumer hardware, reduced inference costs, and improved inference speeds making deployment more        │
│  practical for businesses and individual developers."],"summary":"The large language model landscape is         │
│  rapidly evolving with three dominant trends: multimodal capabilities that integrate vision and language,       │
│  explosive growth in open-source models challenging proprietary systems, and increasing focus on efficiency     │
│  through model compression and edge deployment. These trends reflect the industry's push toward more capable,   │
│  accessible, and practical AI systems."}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
report = structured_result.pydantic

print(f'Topic: {report.topic}')
print(f'Summary: {report.summary}')
for i, trend in enumerate(report.trends, 1):
    print(f'  {i}. {trend}')

Topic: Large Language Models
Summary: The large language model landscape is rapidly evolving with three dominant trends: multimodal capabilities that integrate vision and language, explosive growth in open-source models challenging proprietary systems, and increasing focus on efficiency through model compression and edge deployment. These trends reflect the industry's push toward more capable, accessible, and practical AI systems.
  1. Multimodal Integration: LLMs are rapidly expanding beyond text to seamlessly process images, audio, and video, with models like GPT-4V, Gemini, and open-source alternatives demonstrating sophisticated vision-language understanding capabilities that enable new applications in document analysis, visual reasoning, and interactive AI assistants.
  2. Open-Source Competition: The open-source community is closing the gap with proprietary models, with releases like Llama 3, Mixtral, and DBRX offering near state-of-the-art performance while providing transparenc

## 7. Custom Tool Example

Any Python function can become a CrewAI tool with the `@tool` decorator.

In [23]:
@tool("Python Executor")
def execute_python(code: str) -> str:
    """Execute a Python expression and return the result."""
    try:
        return str(eval(code))
    except Exception as e:
        return f'Error: {e}'


math_agent = Agent(
    role='Math Assistant',
    goal='Solve math problems by executing Python code',
    backstory='You are a precise math assistant who always uses the Python Executor tool to compute answers.',
    tools=[execute_python],
    llm=llm,
    verbose=True
)

math_task = Task(
    description='What is the 20th Fibonacci number? Use the Python Executor tool to compute it.',
    expected_output='The 20th Fibonacci number as a single integer.',
    agent=math_agent
)

math_crew = Crew(
    agents=[math_agent],
    tasks=[math_task],
    process=Process.sequential,
    verbose=True
)

In [24]:
math_result = math_crew.kickoff()
print(math_result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2fdafb32-ef23-4e73-bdd4-ee6a29648dc4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the 20th Fibonacci number? Use the Python Executor tool to compute it.                           │
│  ID: eb1ec654-0796-4af6-86d5-df2896e90e5b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│  Task: What is the 20th Fibonacci number? Use the Python Executor tool to compute it.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool python_executor executed with result: Error: invalid syntax (<string>, line 1)...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: python_executor                                                                                          │
│  Args: {'code': 'def fibonacci(n):\n    if n <= 0:\n        return 0\n    elif n == 1 or n == 2:\n              │
│  return 1\n    else:\n        a, b = 1, 1\n        for _ in range(n - 2):\n            a, b = b, a + b...       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  6765                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the 20th Fibonacci number? Use the Python Executor tool to compute it.                           │
│  Agent: Math Assistant                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2fdafb32-ef23-4e73-bdd4-ee6a29648dc4                                                                       │
│  Final Output: 6765                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

6765


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯